# Gold Data for AI: Feature Tables and LLM Context

Raw IoT events are too noisy and too large for direct ML or LLM consumption. A device that sends 1,000 events per hour doesn't give an ML model 1,000 useful signals — most events carry the same operational state. Gold aggregations are different: they are pre-computed, stable features that represent meaningful patterns. This notebook builds a feature table for device health prediction and shows how gold data structures LLM context for natural-language queries about IoT operations.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sys.path.insert(0, str(Path.cwd()))

In [ ]:
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

events = [json.loads(l) for l in open("../data/input/events.jsonl")]
arrow_table = pa.Table.from_pandas(pd.DataFrame(events))
silver_events = catalog.create_table("silver.events", schema=arrow_table.schema)
silver_events.append(arrow_table)
print(f"Silver: {len(events):,} events")

In [ ]:
from helpers import get_spark

spark = get_spark("AIContextualization", gold_warehouse="../data/warehouse_gold")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

silver_df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())
silver_df = silver_df.withColumn("event_ts", F.to_timestamp("time")) \
                     .withColumn("event_date", F.to_date("time"))

# Load device hierarchy
devices = [json.loads(l) for l in open("../data/input/cmdata.jsonl")]
hierarchy = []
for d in devices:
    parent_id = str(d["id"])
    parent_name = d.get("name", parent_id)
    for child_id in d.get("childAssets", []) + d.get("childDevices", []):
        hierarchy.append({"device_id": str(child_id), "group_id": parent_id, "group_name": parent_name})

hierarchy_df = spark.createDataFrame(hierarchy).dropDuplicates(["device_id"])
print(f"Hierarchy entries: {hierarchy_df.count():,}")

## Why Gold for AI

- **Raw events for ML**: 1.3M events, 6 columns each — noise dominates signal, no temporal structure, storage-intensive.
- **Gold aggregations for ML**: one row per device, 10–20 feature columns — stable, interpretable, ready to train on.
- **Feature consistency**: if both training and inference use the same gold snapshot (via Iceberg time travel), features are consistent. This is the key point-in-time correctness requirement.
- **LLM context**: LLMs have context window limits. You cannot pass 1.3M events to an LLM. Gold tables give compact, structured answers that fit in a prompt.
- **Feature store vs. database**: a feature store is a gold table with versioning (time travel), point-in-time lookups, and explicit contracts about freshness and staleness. Iceberg provides all of this natively.

## Building a Feature Table

A feature table has one row per entity (device) and one column per feature. Features capture patterns in the data: how active is this device? What types of events does it generate? Has it had alarms recently?

The goal is to compress the full history of a device into a small set of meaningful numbers that an ML model can consume directly — without re-reading silver data at inference time.

In [ ]:
# Feature 1: total event count (overall activity level)
total_events = silver_df.groupBy("source").agg(
    F.count("*").alias("total_events")
)

# Feature 2: event type diversity (how many distinct event types)
type_diversity = silver_df.groupBy("source").agg(
    F.countDistinct("type").alias("event_type_count")
)

# Feature 3: most common event type
window_type = Window.partitionBy("source").orderBy(F.col("type_count").desc())
top_type = silver_df.groupBy("source", "type").agg(F.count("*").alias("type_count")) \
    .withColumn("rn", F.row_number().over(window_type)).filter(F.col("rn") == 1) \
    .select(F.col("source"), F.col("type").alias("top_event_type"))

# Feature 4: alarm event count
alarm_events = silver_df.withColumn(
    "is_alarm", F.when(F.col("type").contains("Alarm"), 1).otherwise(0)
).groupBy("source").agg(F.sum("is_alarm").alias("alarm_count"))

# Feature 5: days active (how many distinct days the device sent events)
days_active = silver_df.groupBy("source").agg(
    F.countDistinct("event_date").alias("days_active")
)

# Feature 6: recency (hours since last event)
last_event = silver_df.groupBy("source").agg(F.max("event_ts").alias("last_event_time"))
last_event = last_event.withColumn(
    "hours_since_last_event",
    (F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("last_event_time")) / 3600
)

print("Individual feature DFs ready.")

In [ ]:
# Join all features into one table
features = total_events \
    .join(type_diversity, "source") \
    .join(top_type, "source") \
    .join(alarm_events, "source") \
    .join(days_active, "source") \
    .join(last_event.select("source", "last_event_time", "hours_since_last_event"), "source") \
    .join(hierarchy_df.select(F.col("device_id").alias("source"), "group_name"), "source", "left") \
    .withColumn("group_name", F.coalesce("group_name", F.lit("ungrouped")))

# Derived feature: alarm rate (safe to compute here — at device level, not rolling up)
features = features.withColumn(
    "alarm_rate_pct",
    F.round(100.0 * F.col("alarm_count") / F.col("total_events"), 2)
)

features.writeTo("local.gold.device_features").createOrReplace()
print(f"Feature table rows: {features.count():,}")
features.show(5, truncate=True)

## Point-in-Time Correctness

When training a model, features must reflect what was *known at the time of the label*, not what came later.

Example: training a "will this device have an alarm in the next hour?" model. Features must be computed from events *before* the label timestamp, not including the hour we're predicting. If training features include future data, the model learns patterns it cannot see at inference time — a form of data leakage.

Iceberg time travel solves this:

```python
spark.read.format("iceberg").option("snapshot-id", training_snapshot_id).load(path)
```

This reads the feature table as it existed at a specific snapshot — perfect for point-in-time feature retrieval. Record the snapshot ID alongside each training run so you can reproduce the exact feature set used.

In [ ]:
# The current snapshot is our "as of" point
current_snapshot_id = None
for row in spark.sql("SELECT snapshot_id FROM local.gold.device_features.history").collect():
    current_snapshot_id = row["snapshot_id"]
    break

print(f"Feature table snapshot for training: {current_snapshot_id}")

# Simulate reading features as of this snapshot (e.g., for a training job)
features_at_snapshot = spark.read.format("iceberg") \
    .option("snapshot-id", current_snapshot_id) \
    .load("../data/warehouse_gold/gold/device_features")

print(f"Features at snapshot {current_snapshot_id}: {features_at_snapshot.count():,} devices")

## Device Health Score

A health score is a simple derived metric — no ML framework needed. Combine alarm rate and activity level into a 0–100 score. This demonstrates that gold data enables derived metrics without re-reading silver.

Health scores are useful for:
- Prioritizing which devices to inspect
- Alerting when a previously healthy device degrades
- Feeding as a feature into a predictive maintenance model

In [ ]:
# Simple health score: 100 - weighted penalty for alarms and inactivity
# Scale: 100 = healthy (no alarms, active recently), 0 = critical (many alarms, inactive)
health_scores = features.withColumn(
    "health_score",
    F.greatest(
        F.lit(0.0),
        F.lit(100.0)
        - F.col("alarm_rate_pct") * 2          # each alarm percent costs 2 points
        - F.least(F.lit(50.0), F.col("hours_since_last_event") * 0.5)  # inactivity penalty (max 50 pts)
    ).cast("double")
).select("source", "group_name", "total_events", "alarm_rate_pct", "hours_since_last_event", "health_score")

health_scores.writeTo("local.gold.device_health").createOrReplace()

# Top 10 least healthy devices
print("Least healthy devices:")
spark.sql("""
    SELECT source, group_name, alarm_rate_pct, health_score
    FROM local.gold.device_health
    ORDER BY health_score ASC
    LIMIT 10
""").show(truncate=False)

## Gold Data for LLM Context

Consider the use case: an operator asks "Why was device 140673 unhealthy last week?" An LLM needs context to answer. The options are:

1. Pass raw events — impractical. Millions of rows, huge token cost, and most events contain no useful diagnostic information.
2. Pass gold features — compact, structured, directly answerable.

Gold tables give the LLM exactly what it needs: structured summaries that fit in a context window. This is a form of Retrieval-Augmented Generation (RAG) where the "retrieval" step queries gold tables instead of embedding stores.

The key insight: gold tables are already the right granularity for human-level questions. "What was the alarm rate for device X this week?" maps directly to one gold row.

In [ ]:
def device_summary_text(row) -> str:
    """
    Convert a device health gold row into a human-readable paragraph suitable for LLM context.
    """
    device_id = row["source"]
    group = row["group_name"]
    total = row["total_events"]
    alarm_rate = row["alarm_rate_pct"]
    hours_since = row["hours_since_last_event"]
    score = row["health_score"]

    recency = (
        f"{hours_since:.0f} hours ago"
        if hours_since < 72
        else f"{hours_since / 24:.0f} days ago"
    )

    status = "critical" if score < 30 else ("degraded" if score < 60 else "healthy")
    alarm_desc = (
        "no alarm events" if alarm_rate == 0
        else f"alarm events in {alarm_rate:.1f}% of its events"
    )

    return (
        f"Device {device_id} (group: {group}) has a health score of {score:.0f}/100 ({status}). "
        f"It sent {total:,} events in total, with {alarm_desc}. "
        f"Its most recent event was {recency}."
    )

# Generate summaries for the top 5 least healthy devices
unhealthy = health_scores.orderBy("health_score").limit(5).collect()
print("Device summaries for LLM context:\n")
for row in unhealthy:
    print(device_summary_text(row))
    print()

In [ ]:
# RAG sketch: given a user question, retrieve relevant gold rows, format context, call LLM
def get_rag_context(spark, question: str, device_id: str = None, group_name: str = None) -> str:
    """
    Build LLM context from gold tables based on the query parameters.
    In production: use vector search or query planning to identify relevant tables.
    Here: simple filter-based retrieval.
    """
    contexts = []

    if device_id:
        rows = health_scores.filter(F.col("source") == device_id).collect()
        for row in rows:
            contexts.append(device_summary_text(row))

    if group_name:
        rows = health_scores.filter(F.col("group_name") == group_name).collect()
        for row in rows[:5]:  # Limit to 5 devices per group to stay within context window
            contexts.append(device_summary_text(row))

    if not contexts:
        # Fall back to top unhealthy devices
        rows = health_scores.orderBy("health_score").limit(3).collect()
        for row in rows:
            contexts.append(device_summary_text(row))

    return "\n".join(contexts)

# Example: build context for a specific device query
context = get_rag_context(spark, "Why is device 140673 unhealthy?", device_id="140673")
print("RAG context that would be passed to an LLM:\n")
print(context)
print("\n---")
print("Prompt would be:")
print(f"Context:\n{context}\n\nQuestion: Why is device 140673 unhealthy?\nAnswer:")

## Review Questions

1. What is point-in-time correctness and why does it matter for ML training? How does Iceberg snapshot time travel help?
2. A feature table stores `avg_events_per_hour` per device. You want to roll it up to the group level. Is this safe? What should the table store instead?
3. What is the difference between a feature store and a database table? What capabilities does Iceberg provide that a standard database table might not?
4. What challenges arise when an LLM needs to answer questions about very recent (< 1 hour) IoT events that haven't yet been aggregated into the gold layer?
5. How would you design a gold table optimized for answering "which device groups had the most alarms this month?" Sketch the schema.

## Challenges

- Build a feature table with at least 7 features per device including at least one time-series feature (e.g., trend over the last 7 days vs. the previous 7 days).
- Write a `group_summary_text()` function that summarizes an entire group (using the group_daily gold table from the previous notebook) in 3–5 sentences.
- Design a RAG pipeline for "show me the top 3 device groups with the highest alarm rate this week." What gold tables would you query? How would you format the context?
- Verify point-in-time correctness: append new events to silver, recompute the feature table (creating a new snapshot), then use snapshot time travel to show that the training features (old snapshot) differ from the current features.

## Summary

Key takeaways:

- Gold aggregations are pre-computed features: stable, interpretable, and ready for ML training or inference.
- Point-in-time correctness is critical for ML training: use Iceberg snapshot IDs to retrieve features as they existed at a specific moment in history.
- Health scores and derived metrics can be computed from gold tables without re-reading silver data — the gold layer is the foundation, not just an output.
- LLMs can answer operational questions about IoT devices when given structured gold summaries. Gold tables replace the need to pass millions of raw events to the model.
- The compact, structured nature of gold data makes it ideal for RAG: retrieve the relevant gold rows, format them as text, and include them in the LLM prompt.
- Next: turning these gold patterns into a repeatable, scheduled pipeline.